In [1]:
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
import spikeinterface.comparison as sc
import spikeinterface.exporters as sexp
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
%matplotlib widget


In [2]:
base_folder = Path("C:/Users/jakes/Documents/recording_data/r1299/2023-01-24")
recording = se.read_axona("C:/Users/jakes/Documents/recording_data/r1299/2023-01-24/2023-01-24_r1299_t-maze_2_raw.set")

In [3]:
recording.annotate(is_filtered=False)
channel_ids = recording.get_channel_ids()
fs = recording.get_sampling_frequency()
num_chan = recording.get_num_channels()
num_segments = recording.get_num_segments()

# Invert trace if necessary
#recording = spre.scale(recording, gain=-1)

print(f'Channel ids: {channel_ids}')
print(f'Sampling frequency: {fs}')
print(f'Number of channels: {num_chan}')
print(f"Number of segments: {num_segments}")

Channel ids: ['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15'
 '16' '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29'
 '30' '31']
Sampling frequency: 48000.0
Number of channels: 32
Number of segments: 1


In [4]:
from probeinterface import generate_tetrode, ProbeGroup
probegroup = ProbeGroup()
for i in range(8):
    tetrode = generate_tetrode()
    tetrode.set_device_channel_indices(np.arange(4) + i * 4)
    probegroup.add_probe(tetrode)

recording = recording.set_probegroup(probegroup, group_mode='by_probe')
print(recording.get_channel_groups())

recordings = recording.split_by(property='group', outputs='dict')
print(recordings)

job_kwargs = dict(n_jobs=10, chunk_duration="1s", progress_bar=True)
if (base_folder / "preprocessed").is_dir():
    recording_saved = si.load_extractor(base_folder / "preprocessed")
else:
    recording_saved = recording.save(folder=base_folder / "preprocessed", **job_kwargs)

[0 0 0 0 1 1 1 1 2 2 2 2 3 3 3 3 4 4 4 4 5 5 5 5 6 6 6 6 7 7 7 7]
{0: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 1: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 2: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 3: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 4: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 5: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 6: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s, 7: ChannelSliceRecording: 4 channels - 1 segments - 48.0kHz - 600.006s}


In [ ]:
#Load sorted file if it already exists, otherwise apply a sorting algorithm
if (base_folder / "sorting").is_dir():
    sortings = si.load_extractor(base_folder / "sorting")
else:
    sortings = {}
    for group, sub_recording in recordings.items():
        sorting = ss.run_sorter('mountainsort4', sub_recording, output_folder=f"C:/Users/jakes/Documents/recording_data/r1299/2023-01-24/folder_mountainsort_group{group}", verbose = True, docker_image = True)
        sortings[group] = sorting

sortings

#sorting_klusta = ss.run_klusta(recording_loaded, output_folder = "klusta", docker_image=True, verbose = True)
#sorting_spykingcircus = ss.run_sorter('spykingcircus2', recording_loaded, output_folder = "klusta", verbose = True)

Starting container
Installing spikeinterface==0.96.1 in spikeinterface/mountainsort4-base
Installing extra requirements: ['neo']
Running mountainsort4 sorter inside spikeinterface/mountainsort4-base
Stopping container
Starting container
Installing spikeinterface==0.96.1 in spikeinterface/mountainsort4-base
Installing extra requirements: ['neo']
Running mountainsort4 sorter inside spikeinterface/mountainsort4-base
Stopping container
Starting container
Installing spikeinterface==0.96.1 in spikeinterface/mountainsort4-base
Installing extra requirements: ['neo']
Running mountainsort4 sorter inside spikeinterface/mountainsort4-base
Stopping container
Starting container
Installing spikeinterface==0.96.1 in spikeinterface/mountainsort4-base
Installing extra requirements: ['neo']
Running mountainsort4 sorter inside spikeinterface/mountainsort4-base
Stopping container
Starting container
Installing spikeinterface==0.96.1 in spikeinterface/mountainsort4-base
Installing extra requirements: ['neo']

In [16]:
#Load aggregated sorting from file if available, otherwise aggregate sorted tetrodes and save to disk
if (base_folder / "sorting").is_dir():
    sorting = si.load_extractor(base_folder / "sorting")
    print("Sorting loaded from file")
else:
    for i in list(sortings):
        if i == 0:
            sortings_agg = sortings[0]
        else:
            sortings_agg = si.aggregate_units([sortings_agg, sortings[i]])
    sorting = sortings_agg
    sorting_saved = sorting.save(folder=base_folder / "sorting", **job_kwargs)

In [18]:
print(sorting)
print('HerdingSpikes found', len(sorting.get_unit_ids()), 'units')
sorting = sorting.remove_empty_units()
print(f'HerdingSpikes found {len(sorting.get_unit_ids())} non-empty units')

# for i in range(1,len(sortings)):
#     raster = sw.plot_rasters(sortings[i])
#     plt.savefig(f'C:/Users/jakes/Documents/recording_data/r1299/2023-01-24/plots/raster_tetrode{i}.png')

UnitsSelectionSorting: 18 units - 1 segments - 48.0kHz
HerdingSpikes found 18 units
HerdingSpikes found 18 non-empty units


In [10]:
recording_saved = si.load_extractor(base_folder / "preprocessed")
recording_to_process = recording_saved
recording_f = spre.bandpass_filter(recording_to_process, freq_min=300, freq_max=6000)
recording_cmr = spre.common_reference(recording_f, reference='global', operator='median')
fs = recording_cmr.get_sampling_frequency()


In [11]:
sorting = si.load_extractor(base_folder / "sorting")

we = si.extract_waveforms(recording_f, sorting, folder=base_folder / "waveforms", 
                          load_if_exists=False, overwrite=True)
print(we)


WaveformExtractor: 32 channels - 18 units - 1 segments
  before:144 after:192 n_per_units:500


In [19]:
# import kachery_cloud as kcl
# kcl.init()
waveforms0 = we.get_waveforms(unit_id=0)
print(f"Waveforms shape: {waveforms0.shape}")
template0 = we.get_template(unit_id=0)
print(f"Template shape: {template0.shape}")
all_templates = we.get_all_templates()
print(f"All templates shape: {all_templates.shape}")
correlograms = spost.compute_correlograms(we)
amps = spost.compute_spike_amplitudes(we)
locs = spost.compute_unit_locations(we)
sim = spost.compute_template_similarity(we)
sw.plot_sorting_summary(we, backend='sortingview')


Waveforms shape: (231, 336, 32)
Template shape: (336, 32)
All templates shape: (18, 336, 32)


FigurlFigure(data_uri='sha1://e4eac5e0bff3603e62234f50ac3c5c30cff46563', height=500, view_uri='gs://figurl/spi…

https://figurl.org/f?v=gs://figurl/spikesortingview-10&d=sha1://e4eac5e0bff3603e62234f50ac3c5c30cff46563&label=SpikeInterface%20-%20Sorting%20Summary


In [ ]:
#Export to phy
sexp.export_to_phy(we, output_folder=base_folder / 'phy', 
          compute_amplitudes=True, compute_pc_features=True, copy_binary=True, peak_sign = 'neg', verbose = True,
                   **job_kwargs)

In [50]:
#run phy
import os
os.environ["QTWEBENGINE_CHROMIUM_FLAGS"] = "--single-process"

#To install phy and dependencies if not already installed
# !conda activate si_env
# !pip install PyQt5==5.12.3
# !pip install pyqtwebengine==5.12.1
# !pip install phy --pre --upgrade

#Run phy on current file
%cd {base_folder}
!phy template-gui phy/params.py


C:\Users\jakes\Documents\recording_data\r1299\2023-01-24


21:28:10.834 [E] model:389            Some channels are on the same position, please check the channel positions file.
21:28:10.834 [W] model:667            Skipping spike waveforms that do not exist, they will be extracted on the fly from the raw data as needed.


In [62]:
np.load(str(base_folder)+'\\phy\\channel_positions.npy')
ss.installed_sorters()

RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptk1e33e29\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptpr13b77p\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptwc0h4y7t\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscript30t6gx71\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscript7eb995b9\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptn23nzimx\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptka7t4sbq\script.bat
RUNNING SHELL SCRIPT: C:\Users\jakes\AppData\Local\Temp\tmp_shellscriptj8ifvmga\script.bat


['mountainsort4', 'spykingcircus2', 'tridesclous', 'tridesclous2']